## Adding another Context into the Cyclic ID Algorithm Testing 

### Goal

The goal of this notebook is to test whether adding another context (J) to the cyclic ID algorithm will allow for unidentifiable results to become identifiable ones.

### Approach

1. Test on a simple 5-gene stress response network (where we do know the answers)
2. Compare queries with and without context added
3. Move to full network if necessary


In [1]:
# importing necessary libraries

from y0.graph import NxMixedGraph
from y0.dsl import Variable
from y0.algorithm.identify import Unidentifiable
from y0.algorithm.identify import cyclic_id

print("imported all libraries")



imported all libraries


In [7]:
# Creating the 5-gene network 

nodes = ['fur', 'fnr', 'soxR', 'oxyR', 'soxS']

edges = [
    ('oxyR', 'fur'),
    ('fur', 'soxR'),
    ('soxR', 'oxyR'),
    ('soxR', 'soxS'),
    ('fnr', 'soxR'),
    ('fnr', 'fnr'),
]

stress_graph = NxMixedGraph.from_edges(directed=edges)

print("5-gene network created:")
print(f"  Nodes: {[str(v) for v in stress_graph.nodes()]}")
print(f"  Edges: {len(list(stress_graph.directed.edges()))}")


5-gene network created:
  Nodes: ['oxyR', 'fur', 'soxR', 'soxS', 'fnr']
  Edges: 6


## Test 1 - Query with no context

Now that the network is created, we can now look at a query without its regulated pair to see its result, and the same query with context (J) added.


Query: P(soxR | do(fur))

**Expected result:** Unidentifiable

In [8]:
print("="*70)
print("Query Test 1: P(soxR | do(fur)) - No context added")
print("="*70)

try:
    result = cyclic_id(
        graph=stress_graph,
        outcomes={Variable('soxR')},
        interventions={Variable('fur')}
    )
    print("Result: IDENTIFIABLE")
    print(" UNEXPECTED - should be unidentifiable!")
except Unidentifiable as e:
    print("Result: UNIDENTIFIABLE ✓")
    print(f"Reason: {str(e)[:150]}...")

Query Test 1: P(soxR | do(fur)) - No context added
Result: UNIDENTIFIABLE ✓
Reason: Cannot identify P{soxR} | do{fur})). District frozenset({soxR}) failed identification....


In [9]:

print("\n" + "="*70)
print("Test 1.5: P(soxR | do(fnr))")
print("="*70)
print("Testing the other regulator (fnr) without context")
print()

try:
    result = cyclic_id(
        graph=stress_graph,
        outcomes={Variable('soxR')},
        interventions={Variable('fnr')}
    )
    print("Result: IDENTIFIABLE ✓")
    print("fnr is outside the cycle - works without context")
except Unidentifiable as e:
    print("Result: UNIDENTIFIABLE")
    print(f"Reason: {str(e)[:150]}...")


Test 1.5: P(soxR | do(fnr))
Testing the other regulator (fnr) without context

Result: IDENTIFIABLE ✓
fnr is outside the cycle - works without context


## Test 2 - Query with context added now - fnr

Query: P(soxR | do(fur), J=fnr)

**Setup:**

- Manually remove fnr from graph to simulate the context
- Test P(soxR | do(fur)) on modified graph

In [10]:
print("\n" + "="*70)
print("Test 2: P(soxR | do(fur), J=fnr) - With context (manually before adding code)")
print("="*70)
print("Context: fnr (the other regulator of soxR)")
print()

# Manually remove fnr (context)
stress_graph_no_fnr = stress_graph.remove_nodes_from({Variable('fur')})

print(f"✓ Removed fur from graph (context)")
print(f"  Nodes remaining: {[str(v) for v in stress_graph_no_fnr.nodes()]}")
print()

# test this on a modified graph
try:
    result = cyclic_id(
        graph=stress_graph_no_fnr,
        outcomes={Variable('soxR')},
        interventions={Variable('fnr')}
    )
    print("Result: IDENTIFIABLE ✓")
    print("Context appears to help identifiability.")
except Unidentifiable as e:
    print("Result: Still UNIDENTIFIABLE")
    print(f"Reason: {str(e)[:150]}...")
    print("\nContext didn't help - fur and soxR still in cycle together")


Test 2: P(soxR | do(fur), J=fnr) - With context (manually before adding code)
Context: fnr (the other regulator of soxR)

✓ Removed fur from graph (context)
  Nodes remaining: ['soxS', 'soxR', 'oxyR', 'fnr']

Result: IDENTIFIABLE ✓
Context appears to help identifiability.


In [14]:

print("\n" + "="*70)
print("Test 3: P(soxR | do(fur), J=fnr) - Control test to show that removing a node not in the cycle does not make the intervention identifiable")
print("="*70)
print("Context: fnr (is not in the same 3-gene cycle)")
print()

stress_graph_no_fnr = stress_graph.remove_nodes_from({Variable('fnr')})

print(f"✓ Removed fnr from graph (context)")
print(f"  Nodes remaining: {[str(v) for v in stress_graph_no_fnr.nodes()]}")
print()

try:
    result = cyclic_id(
        graph=stress_graph_no_fnr,
        outcomes={Variable('soxR')},
        interventions={Variable('fur')}
    )
    print("Result: IDENTIFIABLE ✓")
    print("Context helped!")
except Unidentifiable as e:
    print("Result: Still UNIDENTIFIABLE")
    print(f"Reason: {str(e)[:150]}...")
    print("\n✓ Expected: Removing fnr doesn't break the fur-soxR-oxyR cycle")


Test 3: P(soxR | do(fur), J=fnr) - Control test to show that removing a node not in the cycle does not make the intervention identifiable
Context: fnr (is not in the same 3-gene cycle)

✓ Removed fnr from graph (context)
  Nodes remaining: ['soxS', 'soxR', 'fur', 'oxyR']

Result: Still UNIDENTIFIABLE
Reason: Cannot identify P{soxR} | do{fur})). District frozenset({soxR}) failed identification....

✓ Expected: Removing fnr doesn't break the fur-soxR-oxyR cycle


## Summary: Smaller 5-gene network results
---
**Key Findings from testing:**
- P(soxR | do(fur)) → UNIDENTIFIABLE (3-gene feedback loop) with no context added.
- P(soxR | do(fnr), J=fur) → IDENTIFIABLE (removing fur breaks the cycle)
- P(soxR | do(fur), J=fnr) → UNIDENTIFIABLE (fnr removal doesn't help the query become Identifiable)

**Conclusion:** Context can break feedback loops, but only when targeting the right nodes it seems.

---

## Part 2: Testing on Full E. coli Network

Now we test on the full 2,976-gene E. coli regulatory network with biologically relevant dual regulator pairs using the Supplementary table provided.

In [2]:
# loading E. coli network and experimental supplementary tables

import networkx as nx
import pandas as pd

print("="*70)
print("Loading Full Network and Experimental Data")
print("="*70)

# Load full network
ecoli_full_graph = nx.read_graphml("ecoli_full_network_no_small_rna.graphml")

print(f"Full E. coli network:")
print(f"  Nodes: {ecoli_full_graph.number_of_nodes():,}")
print(f"  Edges: {ecoli_full_graph.number_of_edges():,}")

# Load supplementary table with perturbed TFs
df_supp = pd.read_csv('supptable1.csv')

# Filter to high-throughput single-gene perturbations (exclude controls and off-targets)
df_ht = df_supp[df_supp['Experiment'].str.contains('High-throughput pooled library', na=False)]
df_ht = df_ht[~df_ht['Perturbation Name'].str.contains('off|control', case=False, na=False)]

# Get unique perturbed TFs
perturbed_tfs = set(df_ht['Perturbation Name'].str.lower().unique())

print(f"\nPerturbed TFs from experiments: {len(perturbed_tfs)}")
print(f"Including hub regulators: {sorted([tf for tf in perturbed_tfs if tf in ['crp', 'fnr', 'arcA', 'hns', 'lrp', 'rpos']])}")

Loading Full Network and Experimental Data
Full E. coli network:
  Nodes: 2,976
  Edges: 9,211

Perturbed TFs from experiments: 52
Including hub regulators: ['crp', 'fnr', 'hns', 'lrp', 'rpos']


In [3]:
# Find dual regulator pairs 

from itertools import combinations

print("="*70)
print("Finding Dual Regulator Pairs from Experimental Data")
print("="*70)

# Find dual regulator pairs where BOTH TFs were perturbed
dual_pairs_experimental = []

for target in ecoli_full_graph.nodes():
    target_lower = target.lower()
    
    # Get regulators
    regulators = list(ecoli_full_graph.predecessors(target))
    reg_lower = [r.lower() for r in regulators]
    
    # Keep only perturbed regulators
    perturbed_regs = [r for r in reg_lower if r in perturbed_tfs]
    
    if len(perturbed_regs) >= 2:
        for tf1, tf2 in combinations(perturbed_regs, 2):
            dual_pairs_experimental.append({
                'target': target_lower,
                'tf1': tf1,
                'tf2': tf2,
                'total_regulators': len(regulators),
                'perturbed_regulators': len(perturbed_regs)
            })

# Convert to dataframe
df_exp_pairs = pd.DataFrame(dual_pairs_experimental)

print(f"Total dual regulator pairs (both TFs perturbed): {len(df_exp_pairs)}")
print(f"Unique targets: {df_exp_pairs['target'].nunique()}")

# Focus on SIMPLER cases (2-4 regulators)
print(f"\n{'='*70}")
print("SIMPLER CASES (2-4 total regulators)")
print("="*70)

simple_pairs = df_exp_pairs[df_exp_pairs['total_regulators'] <= 4].sort_values('total_regulators')

print(f"Pairs with 2-4 regulators: {len(simple_pairs)}")
print(f"\nFirst 20 examples:")
print(simple_pairs.head(20).to_string(index=False))

# Show distribution
print(f"\n{'='*70}")
print("Distribution by regulator count:")
print("="*70)
for n in range(2, 12):
    count = len(df_exp_pairs[df_exp_pairs['total_regulators'] == n])
    if count > 0:
        print(f"  {n} regulators: {count} pairs")

Finding Dual Regulator Pairs from Experimental Data
Total dual regulator pairs (both TFs perturbed): 1925
Unique targets: 761

SIMPLER CASES (2-4 total regulators)
Pairs with 2-4 regulators: 327

First 20 examples:
target  tf1  tf2  total_regulators  perturbed_regulators
  sodc rpos yeie                 2                     2
  yqef  lrp  crp                 2                     2
  yfcc argr  lrp                 2                     2
  rava  fnr rpos                 2                     2
  viaa  fnr rpos                 2                     2
  fbab rpos  cra                 2                     2
  yjbr gade  lrp                 2                     2
  ldcc rpos glar                 2                     2
  pdeg  hns glar                 2                     2
  argp phob argp                 2                     2
  rssb rpos glar                 2                     2
  ycjp glar ycjw                 2                     2
  ydck  lrp glar                 2          

In [15]:
# Test simpler dual regulator pairs (corrected with node mapping and self-regulation handling)

from y0.graph import NxMixedGraph
from y0.dsl import Variable
from y0.algorithm.identify import cyclic_id, Unidentifiable

print("="*70)
print("Testing Simpler Dual Regulator Pairs")
print("="*70)

# Convert full network to NxMixedGraph
ecoli_full_edges = list(ecoli_full_graph.edges())
ecoli_full_mixed = NxMixedGraph.from_edges(directed=ecoli_full_edges)

print(f"Converted to NxMixedGraph: {len(ecoli_full_mixed.nodes())} nodes")

# Create mapping: lowercase -> actual Variable in graph
node_mapping = {}
for node in ecoli_full_mixed.nodes():
    node_mapping[str(node).lower()] = node

print(f"Created node mapping for {len(node_mapping)} nodes")

# Test a sample of simpler cases (2-3 regulators)
test_sample = simple_pairs[simple_pairs['total_regulators'] <= 3].head(20)

print(f"\nTesting pairs with 2-3 regulators:")
print("="*70)

results = []
tested = 0

for idx, row in test_sample.iterrows():
    # Check if all nodes exist in graph
    if (row['target'] not in node_mapping or 
        row['tf1'] not in node_mapping or 
        row['tf2'] not in node_mapping):
        continue
    
    # Get actual Variable objects from graph
    target_var = node_mapping[row['target']]
    tf1_var = node_mapping[row['tf1']]
    tf2_var = node_mapping[row['tf2']]
    
    # Test without context
    try:
        result = cyclic_id(
            graph=ecoli_full_mixed,
            outcomes={target_var},
            interventions={tf1_var}
        )
        status_alone = "IDENTIFIABLE"
    except Unidentifiable:
        status_alone = "UNIDENTIFIABLE"
    
    # Test with context (if it was unidentifiable alone)
    status_with_context = "N/A"
    if status_alone == "UNIDENTIFIABLE":
        # Skip if removing context would remove the target (self-regulation case)
        if row['tf2'] == row['target']:
            status_with_context = "SKIPPED (self-regulation)"
        else:
            # Remove tf2 as context
            graph_no_tf2 = ecoli_full_mixed.remove_nodes_from({tf2_var})
            try:
                result = cyclic_id(
                    graph=graph_no_tf2,
                    outcomes={target_var},
                    interventions={tf1_var}
                )
                status_with_context = "IDENTIFIABLE ✓"
            except (Unidentifiable, ValueError):
                status_with_context = "UNIDENTIFIABLE"
    
    results.append({
        'target': row['target'],
        'tf1': row['tf1'],
        'tf2': row['tf2'],
        'total_regs': row['total_regulators'],
        'alone': status_alone,
        'with_context': status_with_context
    })
    
    tested += 1
    
    symbol = "✓" if status_alone == "IDENTIFIABLE" else "✗"
    print(f"{symbol} P({row['target']} | do({row['tf1']})): {status_alone}", end="")
    if status_with_context != "N/A":
        print(f" → with J={row['tf2']}: {status_with_context}")
    else:
        print()
    
    if tested >= 15:  # Stop after 15 valid tests , can change later. 
        break

# Summary
print(f"\n{'='*70}")
print("FINAL SUMMARY - FULL NETWORK")
print("="*70)

identifiable = sum(1 for r in results if r['alone'] == 'IDENTIFIABLE')
unidentifiable = sum(1 for r in results if r['alone'] == 'UNIDENTIFIABLE')
context_helped = sum(1 for r in results if r['with_context'] == 'IDENTIFIABLE ✓')

print(f"Pairs tested: {len(results)}")
print(f"Identifiable alone: {identifiable}/{len(results)}")
print(f"Unidentifiable alone: {unidentifiable}/{len(results)}")
if unidentifiable > 0:
    print(f"Context made identifiable: {context_helped}/{unidentifiable}")
    
if context_helped > 0:
    print(f"\n{'='*70}")
    print("✓ Success Cases: Adding J parameter works on these example cases")
    print("="*70)
    print(f"Found {context_helped} cases where context breaks feedback loops.")
   

Testing Simpler Dual Regulator Pairs
Converted to NxMixedGraph: 2962 nodes
Created node mapping for 2962 nodes

Testing pairs with 2-3 regulators:
✗ P(sodc | do(rpos)): UNIDENTIFIABLE → with J=yeie: IDENTIFIABLE ✓
✗ P(yqef | do(lrp)): UNIDENTIFIABLE → with J=crp: IDENTIFIABLE ✓
✗ P(yfcc | do(argr)): UNIDENTIFIABLE → with J=lrp: IDENTIFIABLE ✓
✗ P(rava | do(fnr)): UNIDENTIFIABLE → with J=rpos: IDENTIFIABLE ✓
✗ P(viaa | do(fnr)): UNIDENTIFIABLE → with J=rpos: IDENTIFIABLE ✓
✗ P(fbab | do(rpos)): UNIDENTIFIABLE → with J=cra: IDENTIFIABLE ✓
✗ P(yjbr | do(gade)): UNIDENTIFIABLE → with J=lrp: IDENTIFIABLE ✓
✓ P(ldcc | do(rpos)): IDENTIFIABLE
✓ P(pdeg | do(hns)): IDENTIFIABLE
✗ P(argp | do(phob)): UNIDENTIFIABLE → with J=argp: SKIPPED (self-regulation)
✓ P(rssb | do(rpos)): IDENTIFIABLE
✓ P(ycjp | do(glar)): IDENTIFIABLE
✓ P(ydck | do(lrp)): IDENTIFIABLE
✓ P(ygiv | do(lrp)): IDENTIFIABLE
✓ P(ygiw | do(lrp)): IDENTIFIABLE

FINAL SUMMARY - FULL NETWORK
Pairs tested: 15
Identifiable alone: 7/15
